# 10. AutoGaze 범용 벤치마크 — ViT / MLLM 통합 가이드

> AutoGaze를 다양한 ViT 및 MLLM 모델에 적용하여 **토큰 효율성**, **지연 시간**, **품질**을 측정하는 방법을 실습합니다.

## 목차
1. 환경 설정 (폰트 · 디바이스)
2. 지원 모델 레지스트리
3. AutoGaze 로드
4. 통합 방식 A — Zero-shot (`AutoGazeTokenSelector`)
5. DINOv2 벤치마크
6. YOLOS 벤치마크
7. Depth-Anything-V2 벤치마크
8. SigLIP 벤치마크
9. 모델별 지연 시간 비교
10. Gazing Ratio vs 품질 트레이드오프
11. 새 모델 추가 방법 (템플릿)
12. 요약

---
## 1. 환경 설정

In [ ]:
import sys, os, warnings, time
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# ── 한국어 폰트 설정 ──────────────────────────────────────────────
def _configure_mpl_cjk():
    matplotlib.rcParams["axes.unicode_minus"] = False
    try:
        import matplotlib.font_manager as fm
        for name in ["Apple SD Gothic Neo", "AppleGothic", "NanumGothic",
                     "Noto Sans CJK KR", "Noto Sans CJK JP", "DejaVu Sans"]:
            try:
                path = fm.findfont(fm.FontProperties(family=name), fallback_to_default=False)
                if path and os.path.exists(path):
                    matplotlib.rcParams["font.family"] = name
                    return name
            except Exception:
                pass
    except Exception:
        pass
    return "default"

font_name = _configure_mpl_cjk()

# ── 디바이스 설정 ─────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

def _sync():
    if device.type == "cuda":  torch.cuda.synchronize()
    elif device.type == "mps": torch.mps.synchronize()

print(f"device : {device}")
print(f"font   : {font_name}")

---
## 2. 지원 모델 레지스트리

아래 딕셔너리에 모델 ID, 임베딩 모듈 경로, 패치 크기를 등록합니다.  
새 모델을 추가하려면 항목을 하나만 추가하면 됩니다.

In [ ]:
# MODEL_REGISTRY 구조:
#   embed_path  : model.named_modules() 상의 임베딩 서브모듈 경로
#   patch_size  : 픽셀 단위 패치 크기
#   img_size    : 기본 입력 해상도
#   has_cls     : [CLS] 토큰 유무
#   loader      : (model_id) -> (model, processor) 를 반환하는 함수명

MODEL_REGISTRY = {
    "DINOv2-base": {
        "model_id"  : "facebook/dinov2-base",
        "embed_path": "embeddings",
        "patch_size": 14,
        "img_size"  : 224,
        "has_cls"   : True,
        "loader"    : "load_dinov2",
    },
    "YOLOS-tiny": {
        "model_id"  : "hustvl/yolos-tiny",
        "embed_path": "vit.embeddings",
        "patch_size": 16,
        "img_size"  : 224,
        "has_cls"   : True,
        "loader"    : "load_yolos",
    },
    "Depth-Anything-V2": {
        "model_id"  : "depth-anything/Depth-Anything-V2-Small-hf",
        "embed_path": "backbone.embeddings",
        "patch_size": 14,
        "img_size"  : 224,
        "has_cls"   : True,
        "loader"    : "load_depth_anything",
    },
    "SigLIP-base/16": {
        "model_id"  : "google/siglip-base-patch16-224",
        "embed_path": "vision_model.embeddings",
        "patch_size": 16,
        "img_size"  : 224,
        "has_cls"   : False,   # SigLIP은 CLS 토큰 없음
        "loader"    : "load_siglip",
    },
}

def get_grid(cfg):
    """패치 그리드 크기 (h, w) 반환."""
    g = cfg["img_size"] // cfg["patch_size"]
    return g, g

def get_submodule(model, path):
    """점 표기법 경로로 서브모듈 반환. e.g. 'vit.embeddings'"""
    m = model
    for part in path.split("."):
        m = getattr(m, part)
    return m

print("등록된 모델:", list(MODEL_REGISTRY.keys()))

---
## 3. AutoGaze 로드

`weights/AutoGaze` 디렉토리에서 AutoGaze 가중치를 로드합니다.  
가중치가 없으면 **데모 모드**로 전환되어 랜덤 gaze mask를 사용합니다.

In [ ]:
from autogaze.models.autogaze import AutoGaze
from autogaze.models.autogaze.autogaze_cv import AutoGazeTokenSelector
from autogaze.models.autogaze.processing_autogaze import AutoGazeImageProcessor

AG_PATH   = "../weights/AutoGaze"
DEMO_MODE = False

try:
    ag_model = AutoGaze.from_pretrained(AG_PATH).eval().to(device)
    ag_proc  = AutoGazeImageProcessor.from_pretrained(AG_PATH)
    print(f"AutoGaze 로드 완료: {AG_PATH}")
    print(f"  파라미터: {sum(p.numel() for p in ag_model.parameters()):,}")
except Exception as e:
    print(f"[경고] AutoGaze 가중치 없음 → 데모 모드 ({e})")
    DEMO_MODE = True
    ag_model  = None
    ag_proc   = None

print(f"DEMO_MODE: {DEMO_MODE}")

In [ ]:
# ── AutoGazeTokenSelector 또는 데모용 Mock 준비 ──────────────────

class MockSelector:
    """실제 AutoGaze 없이 랜덤 gaze mask를 생성하는 데모용 클래스."""
    def __init__(self, gazing_ratio=0.5):
        self.gazing_ratio = gazing_ratio

    def compute_gaze_mask(self, ag_video, target_h, target_w, threshold=0.5):
        B = ag_video.shape[0]
        N = target_h * target_w
        k = max(1, int(N * self.gazing_ratio))
        mask = torch.zeros(B, N, dtype=torch.bool, device=ag_video.device)
        for b in range(B):
            idx = torch.randperm(N)[:k]
            mask[b, idx] = True
        return mask

    def token_mask_context(self, embed_module, mask, has_cls_token=True):
        # AutoGazeTokenSelector의 context manager를 재현
        from contextlib import contextmanager
        @contextmanager
        def _ctx():
            _mask = mask
            N_mask = mask.shape[-1]
            def _hook(module, inp, out):
                B = out.shape[0]
                if has_cls_token:
                    prefix = out[:, :1, :]
                    rest   = out[:, 1:, :]
                    if rest.shape[1] > N_mask:
                        patches = rest[:, :N_mask, :]
                        suffix  = rest[:, N_mask:, :]
                        w = _mask[:B].float().unsqueeze(-1)
                        return torch.cat([prefix, patches * w, suffix], dim=1)
                    else:
                        w = _mask[:B, :rest.shape[1]].float().unsqueeze(-1)
                        return torch.cat([prefix, rest * w], dim=1)
                else:
                    w = _mask[:B, :out.shape[1]].float().unsqueeze(-1)
                    return out * w
            handle = embed_module.register_forward_hook(_hook)
            try:
                yield
            finally:
                handle.remove()
        return _ctx()

if DEMO_MODE:
    selector = MockSelector(gazing_ratio=0.5)
    print("MockSelector 준비 완료 (데모 모드)")
else:
    selector = AutoGazeTokenSelector(ag_model, gazing_ratio=0.5)
    print("AutoGazeTokenSelector 준비 완료")

---
## 4. 통합 방식 A — Zero-shot (`AutoGazeTokenSelector`)

AutoGaze가 예측한 14×14 gaze map을 타겟 ViT의 패치 그리드로 보간하여 중요하지 않은 패치를 0으로 마스킹합니다.  
**모델 구조 수정 없이** 어떤 ViT에도 적용 가능합니다.

In [ ]:
# ── 통합 방식 다이어그램 시각화 ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# 왼쪽: AutoGaze 14×14 gaze map
np.random.seed(42)
gaze_map = np.random.rand(14, 14)
gaze_map[5:9, 5:9] += 1.5  # 중앙 영역 강조
gaze_map = np.clip(gaze_map / gaze_map.max(), 0, 1)
axes[0].imshow(gaze_map, cmap="hot", vmin=0, vmax=1)
axes[0].set_title("AutoGaze\n14×14 Gaze Map", fontsize=12)
axes[0].axis("off")

# 가운데: 보간 → 타겟 그리드
from scipy.ndimage import zoom
target_map = zoom(gaze_map, 224/14/16, order=1)  # 16×16 그리드
target_map = (target_map > 0.5).astype(float)
axes[1].imshow(target_map, cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("보간 후\n16×16 Bool Mask", fontsize=12)
# 그리드 선
for i in range(1, 16):
    axes[1].axhline(i * 224/16/224 * target_map.shape[0], color='gray', lw=0.5, alpha=0.5)
    axes[1].axvline(i * 224/16/224 * target_map.shape[1], color='gray', lw=0.5, alpha=0.5)
axes[1].axis("off")

# 오른쪽: 선택 비율에 따른 토큰 수
ratios = np.linspace(0.1, 1.0, 10)
models = {"DINOv2 (256)": 256, "ViT-B/16 (196)": 196, "SigLIP (196)": 196}
colors = ["#2196F3", "#4CAF50", "#FF9800"]
for (name, total), c in zip(models.items(), colors):
    axes[2].plot(ratios * 100, ratios * total, "o-", label=name, color=c, lw=2, ms=5)
axes[2].set_xlabel("Gazing Ratio (%)")
axes[2].set_ylabel("선택된 토큰 수")
axes[2].set_title("Ratio별 토큰 수", fontsize=12)
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)
axes[2].set_xticks([10, 25, 50, 75, 100])

fig.suptitle("AutoGaze Zero-shot 통합 원리", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 5. DINOv2 벤치마크

DINOv2-base: patch_size=14, img_size=224 → **16×16 = 256 토큰** + 1 CLS

In [ ]:
from transformers import AutoModel, AutoImageProcessor

cfg_dino = MODEL_REGISTRY["DINOv2-base"]
print(f"모델 로드: {cfg_dino['model_id']}")

dino_model  = AutoModel.from_pretrained(cfg_dino["model_id"]).eval().to(device)
dino_proc   = AutoImageProcessor.from_pretrained(cfg_dino["model_id"])
dino_embed  = get_submodule(dino_model, cfg_dino["embed_path"])
grid_h, grid_w = get_grid(cfg_dino)

print(f"  임베딩 모듈: {cfg_dino['embed_path']}")
print(f"  패치 그리드: {grid_h}×{grid_w} = {grid_h*grid_w} 토큰")
print(f"  파라미터: {sum(p.numel() for p in dino_model.parameters()):,}")

In [ ]:
# ── 샘플 이미지 생성 (실제 이미지 있으면 교체) ─────────────────────
sample_img = Image.fromarray(
    np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
)

# AutoGaze용 입력 (B=1, T=1, C=3, H=224, W=224)
if DEMO_MODE:
    ag_video = torch.randn(1, 1, 3, 224, 224).to(device)
else:
    ag_tensor = ag_proc(images=sample_img, return_tensors="pt")["pixel_values"]
    ag_video  = ag_tensor.unsqueeze(0).unsqueeze(0).to(device)

# DINOv2용 입력
dino_inputs = dino_proc(images=sample_img, return_tensors="pt")
dino_inputs = {k: v.to(device) for k, v in dino_inputs.items()}

# ── Gaze Mask 계산 ─────────────────────────────────────────────
mask_dino = selector.compute_gaze_mask(
    ag_video, target_h=grid_h, target_w=grid_w
)
n_selected = mask_dino[0].sum().item()
print(f"선택 토큰: {n_selected}/{grid_h*grid_w} ({100*n_selected/(grid_h*grid_w):.1f}%)")

# ── Latency 측정 (WARM UP 포함) ───────────────────────────────
N_RUNS = 20

# Warm up
for _ in range(3):
    with torch.no_grad():
        _ = dino_model(**dino_inputs)
_sync()

# Full (AutoGaze 없음)
t_full = []
for _ in range(N_RUNS):
    _sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        out_full = dino_model(**dino_inputs)
    _sync()
    t_full.append((time.perf_counter() - t0) * 1000)

# With AutoGaze
t_ag = []
for _ in range(N_RUNS):
    _sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        with selector.token_mask_context(dino_embed, mask_dino, has_cls_token=cfg_dino["has_cls"]):
            out_ag = dino_model(**dino_inputs)
    _sync()
    t_ag.append((time.perf_counter() - t0) * 1000)

ms_full = np.median(t_full)
ms_ag   = np.median(t_ag)
print(f"\n[DINOv2-base] device={device}")
print(f"  Full      : {ms_full:.2f} ms")
print(f"  AutoGaze  : {ms_ag:.2f} ms  ({100*(ms_full-ms_ag)/ms_full:.1f}% 절감)")

---
## 6. YOLOS 벤치마크

YOLOS-tiny: patch_size=16, img_size=224 → **14×14 = 196 패치 토큰** + CLS + 100 detection tokens

In [ ]:
from transformers import YolosModel, YolosImageProcessor

cfg_yolos = MODEL_REGISTRY["YOLOS-tiny"]
print(f"모델 로드: {cfg_yolos['model_id']}")

yolos_model  = YolosModel.from_pretrained(cfg_yolos["model_id"]).eval().to(device)
yolos_proc   = YolosImageProcessor.from_pretrained(cfg_yolos["model_id"])
yolos_embed  = get_submodule(yolos_model, cfg_yolos["embed_path"])
yolos_h, yolos_w = get_grid(cfg_yolos)

print(f"  임베딩 모듈: {cfg_yolos['embed_path']}")
print(f"  패치 그리드: {yolos_h}×{yolos_w} = {yolos_h*yolos_w} 토큰")
print(f"  파라미터: {sum(p.numel() for p in yolos_model.parameters()):,}")

In [ ]:
yolos_inputs = yolos_proc(images=sample_img, return_tensors="pt")
yolos_inputs = {k: v.to(device) for k, v in yolos_inputs.items()}

if DEMO_MODE:
    ag_video_y = torch.randn(1, 1, 3, 224, 224).to(device)
else:
    ag_video_y = ag_video

mask_yolos = selector.compute_gaze_mask(ag_video_y, target_h=yolos_h, target_w=yolos_w)

# Warm up
for _ in range(3):
    with torch.no_grad(): _ = yolos_model(**yolos_inputs)
_sync()

t_full_y, t_ag_y = [], []
for _ in range(N_RUNS):
    _sync(); t0 = time.perf_counter()
    with torch.no_grad(): _ = yolos_model(**yolos_inputs)
    _sync(); t_full_y.append((time.perf_counter()-t0)*1000)

for _ in range(N_RUNS):
    _sync(); t0 = time.perf_counter()
    with torch.no_grad():
        with selector.token_mask_context(yolos_embed, mask_yolos, has_cls_token=cfg_yolos["has_cls"]):
            _ = yolos_model(**yolos_inputs)
    _sync(); t_ag_y.append((time.perf_counter()-t0)*1000)

ms_full_y = np.median(t_full_y)
ms_ag_y   = np.median(t_ag_y)
print(f"[YOLOS-tiny] device={device}")
print(f"  Full      : {ms_full_y:.2f} ms")
print(f"  AutoGaze  : {ms_ag_y:.2f} ms  ({100*(ms_full_y-ms_ag_y)/ms_full_y:.1f}% 절감)")

---
## 7. Depth-Anything-V2 벤치마크

Depth-Anything-V2-Small: DINOv2 backbone, patch_size=14 → **16×16 = 256 토큰**

In [ ]:
from transformers import AutoModelForDepthEstimation, AutoImageProcessor

cfg_depth = MODEL_REGISTRY["Depth-Anything-V2"]
print(f"모델 로드: {cfg_depth['model_id']}")

depth_model  = AutoModelForDepthEstimation.from_pretrained(cfg_depth["model_id"]).eval().to(device)
depth_proc   = AutoImageProcessor.from_pretrained(cfg_depth["model_id"])
depth_embed  = get_submodule(depth_model, cfg_depth["embed_path"])
depth_h, depth_w = get_grid(cfg_depth)

print(f"  임베딩 모듈: {cfg_depth['embed_path']}")
print(f"  패치 그리드: {depth_h}×{depth_w} = {depth_h*depth_w} 토큰")

In [ ]:
depth_inputs = depth_proc(images=sample_img, return_tensors="pt")
depth_inputs = {k: v.to(device) for k, v in depth_inputs.items()}

ag_video_d = torch.randn(1, 1, 3, 224, 224).to(device) if DEMO_MODE else ag_video
mask_depth = selector.compute_gaze_mask(ag_video_d, target_h=depth_h, target_w=depth_w)

for _ in range(3):
    with torch.no_grad(): _ = depth_model(**depth_inputs)
_sync()

t_full_d, t_ag_d = [], []
for _ in range(N_RUNS):
    _sync(); t0 = time.perf_counter()
    with torch.no_grad(): _ = depth_model(**depth_inputs)
    _sync(); t_full_d.append((time.perf_counter()-t0)*1000)

for _ in range(N_RUNS):
    _sync(); t0 = time.perf_counter()
    with torch.no_grad():
        with selector.token_mask_context(depth_embed, mask_depth, has_cls_token=cfg_depth["has_cls"]):
            _ = depth_model(**depth_inputs)
    _sync(); t_ag_d.append((time.perf_counter()-t0)*1000)

ms_full_d = np.median(t_full_d)
ms_ag_d   = np.median(t_ag_d)
print(f"[Depth-Anything-V2] device={device}")
print(f"  Full      : {ms_full_d:.2f} ms")
print(f"  AutoGaze  : {ms_ag_d:.2f} ms  ({100*(ms_full_d-ms_ag_d)/ms_full_d:.1f}% 절감)")

---
## 8. SigLIP 벤치마크

SigLIP-base/16: patch_size=16 → **14×14 = 196 토큰**, CLS 없음  
(NVILA에서 완전 통합 방식 B로 사용되는 동일 모델)

In [ ]:
from transformers import SiglipModel, SiglipProcessor

cfg_sig = MODEL_REGISTRY["SigLIP-base/16"]
print(f"모델 로드: {cfg_sig['model_id']}")

sig_model  = SiglipModel.from_pretrained(cfg_sig["model_id"]).eval().to(device)
sig_proc   = SiglipProcessor.from_pretrained(cfg_sig["model_id"])
sig_embed  = get_submodule(sig_model, cfg_sig["embed_path"])
sig_h, sig_w = get_grid(cfg_sig)

print(f"  임베딩 모듈: {cfg_sig['embed_path']}")
print(f"  패치 그리드: {sig_h}×{sig_w} = {sig_h*sig_w} 토큰")
print(f"  CLS 토큰: {cfg_sig['has_cls']}")

In [ ]:
sig_inputs = sig_proc(
    images=sample_img, text=["a photo"], return_tensors="pt", padding="max_length"
)
sig_inputs = {k: v.to(device) for k, v in sig_inputs.items()}

ag_video_s = torch.randn(1, 1, 3, 224, 224).to(device) if DEMO_MODE else ag_video
mask_sig = selector.compute_gaze_mask(ag_video_s, target_h=sig_h, target_w=sig_w)

for _ in range(3):
    with torch.no_grad(): _ = sig_model(**sig_inputs)
_sync()

t_full_s, t_ag_s = [], []
for _ in range(N_RUNS):
    _sync(); t0 = time.perf_counter()
    with torch.no_grad(): _ = sig_model(**sig_inputs)
    _sync(); t_full_s.append((time.perf_counter()-t0)*1000)

for _ in range(N_RUNS):
    _sync(); t0 = time.perf_counter()
    with torch.no_grad():
        with selector.token_mask_context(sig_embed, mask_sig, has_cls_token=cfg_sig["has_cls"]):
            _ = sig_model(**sig_inputs)
    _sync(); t_ag_s.append((time.perf_counter()-t0)*1000)

ms_full_s = np.median(t_full_s)
ms_ag_s   = np.median(t_ag_s)
print(f"[SigLIP-base/16] device={device}")
print(f"  Full      : {ms_full_s:.2f} ms")
print(f"  AutoGaze  : {ms_ag_s:.2f} ms  ({100*(ms_full_s-ms_ag_s)/ms_full_s:.1f}% 절감)")

---
## 9. 모델별 지연 시간 비교

방식 A (Zero-shot, forward hook)에서는 **KV cache 절감**은 발생하지만 실제 ViT 연산은 같습니다.  
완전 통합(방식 B)에서는 선택된 토큰만 처리하므로 ViT 자체도 빨라집니다.

In [ ]:
results = {
    "DINOv2-base\n(256 tok)": {"full": ms_full,   "ag": ms_ag,   "tokens": 256, "ratio": selector.gazing_ratio},
    "YOLOS-tiny\n(196 tok)" : {"full": ms_full_y,  "ag": ms_ag_y,  "tokens": 196, "ratio": selector.gazing_ratio},
    "Depth-Anything\n(256 tok)": {"full": ms_full_d, "ag": ms_ag_d, "tokens": 256, "ratio": selector.gazing_ratio},
    "SigLIP-base\n(196 tok)" : {"full": ms_full_s,  "ag": ms_ag_s,  "tokens": 196, "ratio": selector.gazing_ratio},
}

labels = list(results.keys())
full_ms = [results[k]["full"] for k in labels]
ag_ms   = [results[k]["ag"]   for k in labels]
savings = [100 * (f - a) / f for f, a in zip(full_ms, ag_ms)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 절대 latency 비교
x = np.arange(len(labels))
w = 0.35
axes[0].bar(x - w/2, full_ms, w, label="Full (AutoGaze OFF)", color="#607D8B", alpha=0.9)
axes[0].bar(x + w/2, ag_ms,   w, label=f"AutoGaze ON (ratio={selector.gazing_ratio})", color="#2196F3", alpha=0.9)
for i, (f, a) in enumerate(zip(full_ms, ag_ms)):
    axes[0].text(i - w/2, f + 0.3, f"{f:.1f}", ha="center", va="bottom", fontsize=8, color="#607D8B")
    axes[0].text(i + w/2, a + 0.3, f"{a:.1f}", ha="center", va="bottom", fontsize=8, color="#1565C0")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, fontsize=9)
axes[0].set_ylabel("지연 시간 (ms)")
axes[0].set_title(f"모델별 지연 시간 비교 ({device})", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.3)

# 오른쪽: 절감률
bar_colors = ["#4CAF50" if s > 0 else "#F44336" for s in savings]
bars = axes[1].bar(x, savings, color=bar_colors, alpha=0.85)
for bar, s in zip(bars, savings):
    axes[1].text(bar.get_x() + bar.get_width()/2, s + 0.3,
                 f"{s:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, fontsize=9)
axes[1].set_ylabel("지연 시간 절감률 (%)")
axes[1].set_title("AutoGaze ON/OFF 절감률", fontsize=12)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].grid(axis="y", alpha=0.3)

note = ("※ 방식 A (Zero-shot): forward hook으로 비선택 토큰을 0으로 만듦.\n"
        "   ViT 연산 자체는 동일 → 방식 B (완전 통합)에서 더 큰 절감 효과.")
fig.text(0.5, -0.04, note, ha="center", fontsize=9, color="#555", style="italic")
plt.tight_layout()
plt.show()

---
## 10. Gazing Ratio vs 품질 트레이드오프

Gazing ratio를 낮출수록 토큰이 줄지만 특징 복원 품질도 감소합니다.  
DINOv2를 기준으로 여러 ratio에서 특징 유사도(코사인)를 측정합니다.

In [ ]:
ratios_test  = [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 1.0]
cos_sims     = []
token_counts = []
N_TOTAL_DINO = grid_h * grid_w  # 256

# 기준선: 전체 패치 특징
with torch.no_grad():
    out_ref = dino_model(**dino_inputs)
feat_ref = out_ref.last_hidden_state[:, 1:, :]  # CLS 제거 → (1, 256, D)

for r in ratios_test:
    # 해당 ratio로 마스크 생성
    if DEMO_MODE:
        ag_v = torch.randn(1, 1, 3, 224, 224).to(device)
    else:
        ag_v = ag_video

    tmp_sel = MockSelector(gazing_ratio=r) if DEMO_MODE else AutoGazeTokenSelector(ag_model, gazing_ratio=r)
    m = tmp_sel.compute_gaze_mask(ag_v, target_h=grid_h, target_w=grid_w)
    n_tok = m[0].sum().item()
    token_counts.append(n_tok)

    with torch.no_grad():
        with tmp_sel.token_mask_context(dino_embed, m, has_cls_token=cfg_dino["has_cls"]):
            out_ag = dino_model(**dino_inputs)

    feat_ag = out_ag.last_hidden_state[:, 1:, :]  # (1, 256, D)
    sim = F.cosine_similarity(
        feat_ref.reshape(1, -1), feat_ag.reshape(1, -1)
    ).item()
    cos_sims.append(sim)
    print(f"  ratio={r:.2f}  토큰={n_tok}/{N_TOTAL_DINO}  코사인 유사도={sim:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: 코사인 유사도 vs ratio
axes[0].plot(ratios_test, cos_sims, "o-", color="#2196F3", lw=2.5, ms=8)
axes[0].fill_between(ratios_test, cos_sims, alpha=0.15, color="#2196F3")
for r, s in zip(ratios_test, cos_sims):
    axes[0].annotate(f"{s:.3f}", (r, s), textcoords="offset points",
                     xytext=(0, 8), ha="center", fontsize=8)
axes[0].set_xlabel("Gazing Ratio")
axes[0].set_ylabel("코사인 유사도 (vs Full)")
axes[0].set_title("DINOv2: 품질 vs Gazing Ratio", fontsize=12)
axes[0].set_ylim(min(cos_sims) - 0.05, 1.02)
axes[0].set_xticks(ratios_test)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(1.0, color="gray", lw=1, ls="--", label="Full 기준선")
axes[0].legend(fontsize=9)

# 오른쪽: 토큰 수 vs 코사인 유사도 (파레토 곡선)
sc = axes[1].scatter(token_counts, cos_sims,
                     c=ratios_test, cmap="RdYlGn", s=120, zorder=5)
axes[1].plot(token_counts, cos_sims, "--", color="gray", alpha=0.5, lw=1)
for tok, sim, r in zip(token_counts, cos_sims, ratios_test):
    axes[1].annotate(f"r={r}", (tok, sim), textcoords="offset points",
                     xytext=(6, 0), fontsize=8, color="#444")
plt.colorbar(sc, ax=axes[1], label="Gazing Ratio")
axes[1].set_xlabel("선택 토큰 수")
axes[1].set_ylabel("코사인 유사도")
axes[1].set_title("파레토 곡선: 토큰 절감 vs 품질", fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 11. 새 모델 추가 방법 (템플릿)

임의의 ViT 기반 모델을 3단계로 추가할 수 있습니다.

In [ ]:
# ════════════════════════════════════════════════════════════════
# 새 모델 추가 템플릿 — 아래 4개 변수만 수정하면 됩니다.
# ════════════════════════════════════════════════════════════════

NEW_MODEL_ID   = "google/vit-base-patch16-224"   # ← HF model ID
EMBED_PATH     = "vit.embeddings"                 # ← 임베딩 서브모듈 경로
PATCH_SIZE     = 16                               # ← 패치 크기 (px)
IMG_SIZE       = 224                              # ← 입력 이미지 크기
HAS_CLS        = True                             # ← [CLS] 토큰 유무

# ── STEP 1: 모델 로드 ────────────────────────────────────────────
from transformers import ViTModel, ViTImageProcessor
new_model  = ViTModel.from_pretrained(NEW_MODEL_ID).eval().to(device)
new_proc   = ViTImageProcessor.from_pretrained(NEW_MODEL_ID)

# ── STEP 2: 임베딩 모듈 + 패치 그리드 크기 ───────────────────────
new_embed  = get_submodule(new_model, EMBED_PATH)
g = IMG_SIZE // PATCH_SIZE
print(f"패치 그리드: {g}×{g} = {g*g} 토큰")

# ── STEP 3: 벤치마크 실행 ────────────────────────────────────────
new_inputs = new_proc(images=sample_img, return_tensors="pt")
new_inputs = {k: v.to(device) for k, v in new_inputs.items()}

ag_video_new = torch.randn(1, 1, 3, 224, 224).to(device) if DEMO_MODE else ag_video
mask_new = selector.compute_gaze_mask(ag_video_new, target_h=g, target_w=g)

for _ in range(3):
    with torch.no_grad(): _ = new_model(**new_inputs)
_sync()

t_f, t_a = [], []
for _ in range(N_RUNS):
    _sync(); t0 = time.perf_counter()
    with torch.no_grad(): _ = new_model(**new_inputs)
    _sync(); t_f.append((time.perf_counter()-t0)*1000)

for _ in range(N_RUNS):
    _sync(); t0 = time.perf_counter()
    with torch.no_grad():
        with selector.token_mask_context(new_embed, mask_new, has_cls_token=HAS_CLS):
            _ = new_model(**new_inputs)
    _sync(); t_a.append((time.perf_counter()-t0)*1000)

ms_f = np.median(t_f);  ms_a = np.median(t_a)
print(f"[{NEW_MODEL_ID.split('/')[-1]}]")
print(f"  Full     : {ms_f:.2f} ms")
print(f"  AutoGaze : {ms_a:.2f} ms  ({100*(ms_f-ms_a)/ms_f:.1f}% 절감)")
print(f"  선택 토큰: {mask_new[0].sum().item()}/{g*g}")

# ── 레지스트리에 등록 (선택) ─────────────────────────────────────
MODEL_REGISTRY[NEW_MODEL_ID.split('/')[-1]] = {
    "model_id"  : NEW_MODEL_ID,
    "embed_path": EMBED_PATH,
    "patch_size": PATCH_SIZE,
    "img_size"  : IMG_SIZE,
    "has_cls"   : HAS_CLS,
    "loader"    : "manual",
}
print(f"\n레지스트리 등록 완료: {list(MODEL_REGISTRY.keys())}")

---
## 12. 요약

### 통합 방식 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── 방식 비교 테이블 ─────────────────────────────────────────────
table_data = [
    ["구분",         "방식 A: Zero-shot",    "방식 B: 완전 통합"],
    ["모델 수정",     "불필요",               "패치 임베딩 + forward"],
    ["FLOPs 절감",    "없음",                 "있음 (선택 토큰만 계산)"],
    ["지연 절감",     "LLM KV cache 절감",    "ViT + LLM 모두 절감"],
    ["구현 난이도",   "매우 쉬움 (5줄)",       "중간~어려움"],
    ["사용 사례",     "탐색/비교 실험",        "프로덕션 배포"],
    ["참고 구현",     "autogaze_cv.py",        "siglip/modeling_siglip.py"],
]

axes[0].axis("off")
tbl = axes[0].table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    cellLoc="center",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.8)
# 헤더 색
for j in range(3):
    tbl[0, j].set_facecolor("#1565C0")
    tbl[0, j].set_text_props(color="white", fontweight="bold")
# 짝수 행 배경
for i in range(1, len(table_data)):
    for j in range(3):
        if i % 2 == 0:
            tbl[i, j].set_facecolor("#E3F2FD")
axes[0].set_title("방식 A vs 방식 B 비교", fontsize=12, fontweight="bold", pad=12)

# ── 추천 Gazing Ratio ────────────────────────────────────────────
scenarios  = ["실시간\n스트리밍", "균형\n(속도+품질)", "품질 우선\n(긴 비디오)", "기준선\n(AutoGaze OFF)"]
ratio_vals = [0.3, 0.5, 0.75, 1.0]
bar_colors = ["#F44336", "#FF9800", "#4CAF50", "#9E9E9E"]

bars = axes[1].barh(scenarios, ratio_vals, color=bar_colors, alpha=0.85)
for bar, r in zip(bars, ratio_vals):
    axes[1].text(r + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{r:.2f}", va="center", fontsize=11, fontweight="bold")
axes[1].set_xlim(0, 1.15)
axes[1].set_xlabel("Gazing Ratio")
axes[1].set_title("시나리오별 추천 Gazing Ratio", fontsize=12, fontweight="bold")
axes[1].axvline(0.5, color="#1565C0", lw=1.5, ls="--", alpha=0.6, label="균형점")
axes[1].grid(axis="x", alpha=0.3)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\n=== 벤치마크 결과 요약 ===")
all_results = [
    ("DINOv2-base",      ms_full,   ms_ag),
    ("YOLOS-tiny",       ms_full_y, ms_ag_y),
    ("Depth-Anything-V2",ms_full_d, ms_ag_d),
    ("SigLIP-base/16",   ms_full_s, ms_ag_s),
]
print(f"{'모델':<22} {'Full (ms)':>10} {'AutoGaze (ms)':>14} {'절감률':>8}")
print("-" * 58)
for name, f, a in all_results:
    print(f"{name:<22} {f:>10.2f} {a:>14.2f} {100*(f-a)/f:>7.1f}%")

---

## 다음 단계

| 목표 | 방법 |
|------|------|
| **NVILA 종합 벤치마크** | `python scripts/test_nvila.py video.mp4 --compare-autogaze` |
| **gazing ratio 스윕** | `python scripts/test_nvila.py video.mp4 --sweep-ratio` |
| **CV 태스크 품질 평가** | `python scripts/run_cv_tasks.py --input img.jpg --tasks depth yolos dinov2` |
| **MambaGaze 지연 시간 비교** | `python -m mamba_gaze.eval.latency` |
| **하위 태스크 (VideoMME)** | `mamba_gaze/eval/downstream.py` 참고 |

**가이드 문서**: `docs/benchmark_guide.md`